In [8]:
import numpy as np
import chess
import chess.engine
import tensorflow as tf

# Load the saved model
model = tf.keras.models.load_model('hikaru_chess_model_v2.h5')

# Load the move encoder
move_encoder_keys = np.load('move_encoder_classes_v2.npy', allow_pickle=True)
move_encoder = {uci: idx for idx, uci in enumerate(move_encoder_keys)}

In [9]:
# Function to convert board to features
def board_to_features(board):
    features = np.zeros((8, 8, 12), dtype=np.float32)
    for i in range(64):
        piece = board.piece_at(i)
        if piece:
            features[i // 8, i % 8, piece.piece_type - 1 + (6 if piece.color == chess.BLACK else 0)] = 1
    return features

# Function to convert model prediction back to move
def predict_move(board, model, move_encoder):
    features = board_to_features(board)
    features = np.expand_dims(features, axis=0)
    preds = model.predict(features)
    sorted_indices = np.argsort(preds[0])[::-1]  # Indices of moves sorted by probability
    
    for move_idx in sorted_indices:
        move_uci = [uci for uci, idx in move_encoder.items() if idx == move_idx][0]
        move = chess.Move.from_uci(move_uci)
        if move in board.legal_moves:
            return move
    
    return None  # If no legal move is found

# Function to play a game between the model and Stockfish
def play_game_against_stockfish(model, move_encoder, stockfish_path, stockfish_level=1):
    board = chess.Board()
    stockfish = chess.engine.SimpleEngine.popen_uci(stockfish_path)

    while not board.is_game_over():
        if board.turn == chess.WHITE:
            move = predict_move(board, model, move_encoder)
        else:
            result = stockfish.play(board, chess.engine.Limit(time=0.1))
            move = result.move
        
        if move is None:
            print("No legal move found.")
            break
        board.push(move)
    
    stockfish.quit()
    return board.result()

# Function to calculate Elo rating based on game results
def calculate_elo(model, move_encoder, stockfish_path, num_games=10, stockfish_level=1):
    wins, losses, draws = 0, 0, 0

    for _ in range(num_games):
        result = play_game_against_stockfish(model, move_encoder, stockfish_path, stockfish_level)
        if result == '1-0':
            wins += 1
        elif result == '0-1':
            losses += 1
        else:
            draws += 1
    
    print(f"Wins: {wins}, Losses: {losses}, Draws: {draws}")

    # Basic Elo calculation assuming Stockfish has a fixed rating (e.g., 2400 at level 1)
    stockfish_rating = 2400
    K = 32  # K-factor in Elo rating calculation
    score = (wins + 0.5 * draws) / num_games
    expected_score = 1 / (1 + 10 ** ((stockfish_rating - 1500) / 400))  # Assuming initial rating of 1500 for model
    new_rating = 1500 + K * (score - expected_score)

    print(f"Estimated Elo rating for the model: {new_rating}")

In [11]:
if __name__ == "__main__":
    # Path to the Stockfish executable
    stockfish_path = "stockfish/stockfish-windows-x86-64-avx2.exe" 
    # Calculate Elo rating
    calculate_elo(model, move_encoder, stockfish_path, num_games=50, stockfish_level=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━

In [7]:
import numpy as np
import chess
import chess.engine

# Function to play a game between two Stockfish instances
def play_game_stockfish_vs_stockfish(stockfish_path, stockfish_level=1):
    board = chess.Board()
    stockfish1 = chess.engine.SimpleEngine.popen_uci(stockfish_path)
    stockfish2 = chess.engine.SimpleEngine.popen_uci(stockfish_path)
    stockfish1.configure({"Skill Level": stockfish_level})
    stockfish2.configure({"Skill Level": stockfish_level})

    while not board.is_game_over():
        if board.turn == chess.WHITE:
            result = stockfish1.play(board, chess.engine.Limit(time=0.1))
        else:
            result = stockfish2.play(board, chess.engine.Limit(time=0.1))
        board.push(result.move)
    
    stockfish1.quit()
    stockfish2.quit()
    return board.result()

# Function to calculate Elo rating based on game results
def calculate_stockfish_elo(stockfish_path, num_games=50, stockfish_level=1):
    wins, losses, draws = 0, 0, 0

    for _ in range(num_games):
        result = play_game_stockfish_vs_stockfish(stockfish_path, stockfish_level)
        if result == '1-0':
            wins += 1
        elif result == '0-1':
            losses += 1
        else:
            draws += 1
    
    print(f"Wins: {wins}, Losses: {losses}, Draws: {draws}")

    # Basic Elo calculation assuming Stockfish has a fixed rating (e.g., 2400 at level 1)
    stockfish_rating = 2400
    K = 32  # K-factor in Elo rating calculation
    score = (wins + 0.5 * draws) / num_games
    expected_score = 1 / (1 + 10 ** ((stockfish_rating - 2400) / 400))  # Assuming Stockfish plays against itself
    new_rating = 2400 + K * (score - expected_score)

    print(f"Estimated Elo rating for Stockfish: {new_rating}")

if __name__ == "__main__":
    # Path to the Stockfish executable
    stockfish_path = "D:/Downloads_HDD/stockfish-windows-x86-64-avx2/stockfish/stockfish-windows-x86-64-avx2.exe"

    # Calculate Elo rating for Stockfish against itself
    calculate_stockfish_elo(stockfish_path, num_games=50, stockfish_level=1)


Wins: 28, Losses: 22, Draws: 0
Estimated Elo rating for Stockfish: 2401.92
